# CNN multi-branch training + evaluation + save/load demo

This notebook demonstrates:
- Training a multi-branch CNN with data balancing and augmentation
- Evaluating the model on test windows
- Saving and loading the trained model (Keras model + label encoder)

In [ ]:
import importlib
spec = importlib.util.find_spec('tensorflow')
if spec is None:
    print('tensorflow not available - skipping notebook')
else:
    import json
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    from maneuvers.data.loader import generate_synthetic_sequence
    from maneuvers.classify import train_cnn_from_sequence, save_model, load_model

    # generate a slightly longer sequence with more maneuvers
    seq = generate_synthetic_sequence(duration_s=6.0, fs=50, seed=42)

    # train with balancing and augmentation
    model_obj = train_cnn_from_sequence(
        seq,
        model_type="cnn_multi",
        window_s=1.0,
        hop_s=0.5,
        fs=50,
        balance="oversample",
        augment=2,
        epochs=10,
        batch_size=16
    )

    print('Model trained successfully')
    print('Classes:', model_obj['encoder'].classes_)

    # save the model
    save_model(model_obj, 'cnn_multi_model.joblib')
    print('Model saved to cnn_multi_model.joblib')

    # reload and predict on a test window
    loaded = load_model('cnn_multi_model.joblib')
    print('Model loaded successfully')

    # quick evaluation: predict on a single test sequence
    test_seq = generate_synthetic_sequence(duration_s=3.0, fs=50, seed=999)
    from maneuvers.preprocessing import windowed_examples_from_sequence
    X_test, y_test, _ = windowed_examples_from_sequence(test_seq, window_s=1.0, hop_s=0.5, fs=50)
    y_test_enc = loaded['encoder'].transform(y_test)
    preds = loaded['model'].predict(X_test)
    acc = (preds == y_test_enc).mean()
    print(f'Test accuracy on small test set: {acc:.2f}')

    # plot confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test_enc, preds)
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    plt.title('Confusion matrix (test set)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()